In [1]:
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [8]:
df_fc = pd.read_csv('/Users/venello/Repositories/Python/PyCharm/PycharmProjects/JupyterProject/social_data_2026/assignment_1/data/focus_crimes.csv', low_memory=False)
df_fc['incident_date'] = pd.to_datetime(df_fc['incident_date'])
df_fc = df_fc[df_fc['incident_date'].dt.year < 2026]

In [9]:
df_cl = df_fc[df_fc['incident_category'].isin(['motor vehicle theft', 'assault', 'larceny theft',
       'burglary', 'drug offense', 'prostitution', 'homicide'])].copy()

In [71]:
crime_categories = [
    'motor vehicle theft',
    'assault',
    'larceny theft',
    'burglary',
    'drug offense',
    'prostitution',
    'homicide'
]

# Keep only 2015 and 2025
df_temp = df_cl[df_cl['incident_date'].dt.year.isin([2015, 2025])].copy()
df_temp['year'] = df_temp['incident_date'].dt.year

# Count incidents per category for each year
counts = (
    df_temp
    .groupby(['incident_category', 'year'])
    .size()
    .unstack(fill_value=0)
    .reindex(crime_categories, fill_value=0)
)

# Ensure both years exist as columns
for year in [2015, 2025]:
    if year not in counts.columns:
        counts[year] = 0

counts = counts[[2015, 2025]]
counts.columns = ['count_2015', 'count_2025']

# Percent change: 2025 relative to 2015
counts['pct_change'] = ((counts['count_2025'] - counts['count_2015']) / counts['count_2015']) * 100

# Avoid division by zero
counts.loc[counts['count_2015'] == 0, 'pct_change'] = pd.NA

# Bar colors
colors = [
    'seagreen' if pd.notna(v) and v >= 0 else 'crimson'
    for v in counts['pct_change']
]

# Labels
text_labels = [
    f"{v:.1f}%" if pd.notna(v) else "NA"
    for v in counts['pct_change']
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=counts.index,
    y=counts['pct_change'],
    marker_color=colors,
    text=text_labels,
    textposition='outside',
    customdata=counts[['count_2015', 'count_2025']].values,
    hovertemplate=(
        "<b>%{x}</b><br>"
        "2015: %{customdata[0]}<br>"
        "2025: %{customdata[1]}<br>"
        "Percent change: %{y:.1f}%<extra></extra>"
    )
))

fig.add_hline(y=0, line_dash='dash', line_color='black')

fig.update_layout(
    title='Change in crime incidents: 2025 compared with 2015',
    xaxis_title='Incident category',
    yaxis_title='Percent change (%)',
    template='plotly_white',
    height=600
)

fig.show()



In [73]:
import pandas as pd
import plotly.graph_objects as go

# Drug offense only
df_drug = df_fc[df_fc['incident_category'] == 'drug offense'].copy()

# Marijuana-related descriptions
marijuana = [
    'POSSESSION OF MARIJUANA',
    'POSSESSION OF MARIJUANA FOR SALES',
    'SALE OF MARIJUANA',
    'FURNISHING MARIJUANA',
    'TRANSPORTATION OF MARIJUANA',
    'PLANTING/CULTIVATING MARIJUANA',
    'Marijuana Offense',
    'Marijuana, Furnishing',
    'Marijuana, Cultivating/Planting',
    'Marijuana, Sales',
    'Marijuana, Transporting'
]

# ----------------------------
# 1) Drug offense incidents by year
# ----------------------------
yearly = (
    df_drug
    .assign(year=df_drug['incident_date'].dt.year)
    .groupby('year')
    .size()
    .rename('count')
    .sort_index()
    .to_frame()
)

# Year-over-year percentage change
yearly['prev_count'] = yearly['count'].shift(1)
yearly['pct_change'] = ((yearly['count'] - yearly['prev_count']) / yearly['prev_count']) * 100

# Drop first year because it has no previous year
yearly_plot = yearly[yearly.index > yearly.index.min()].copy()
yearly_plot.loc[yearly_plot['prev_count'] == 0, 'pct_change'] = pd.NA

# Shared x-axis for incidents/evolution
x_years = yearly_plot.index.astype(str)

# Incidents view data
line_y = yearly_plot['count'].astype(float)

# Evolution view data
bar_y = [None if pd.isna(v) else float(v) for v in yearly_plot['pct_change']]
bar_text = ["NA" if pd.isna(v) else f"{v:.1f}%" for v in yearly_plot['pct_change']]
bar_colors = ['seagreen' if (v is not None and v >= 0) else 'crimson' for v in bar_y]
bar_custom = yearly_plot[['prev_count', 'count']].fillna(0).to_numpy()

# ----------------------------
# 2) Marijuana rate by year
#    Rate = marijuana-related drug offenses / all drug offenses * 100
# ----------------------------
marijuana_yearly = (
    df_drug[df_drug['incident_description'].isin(marijuana)]
    .assign(year=lambda d: d['incident_date'].dt.year)
    .groupby('year')
    .size()
    .rename('marijuana_count')
    .to_frame()
)

marijuana_plot = yearly[['count']].join(marijuana_yearly, how='left').fillna(0)
marijuana_plot['marijuana_rate_pct'] = (
        marijuana_plot['marijuana_count'] / marijuana_plot['count'] * 100
)

# If you want the marijuana view limited to 2015-2024 like your earlier cell,
# uncomment the next line:
# marijuana_plot = marijuana_plot.loc[2015:2024]

marijuana_x = marijuana_plot.index.astype(str)
marijuana_y = marijuana_plot['marijuana_rate_pct'].astype(float)
marijuana_custom = marijuana_plot[['marijuana_count', 'count']].to_numpy()

# ----------------------------
# 3) Combined figure
# ----------------------------
fig = go.Figure()

# Incidents view
fig.add_trace(go.Scatter(
    x=x_years,
    y=line_y,
    mode='lines+markers',
    name='Drug offense incidents',
    line=dict(color='royalblue', width=2),
    visible=True,
    hovertemplate="<b>%{x}</b><br>Incidents: %{y:.0f}<extra></extra>"
))

# Evolution view
fig.add_trace(go.Bar(
    x=x_years,
    y=bar_y,
    text=bar_text,
    textposition='outside',
    marker=dict(color=bar_colors),
    customdata=bar_custom,
    name='Drug offense evolution',
    visible=False,
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Previous year count: %{customdata[0]:.0f}<br>"
        "Current year count: %{customdata[1]:.0f}<br>"
        "Change: %{y:.1f}%<extra></extra>"
    )
))

# Marijuana rate view
fig.add_trace(go.Scatter(
    x=marijuana_x,
    y=marijuana_y,
    mode='lines+markers',
    name='Marijuana rate',
    line=dict(color='darkgreen', width=2),
    visible=False,
    customdata=marijuana_custom,
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Marijuana incidents: %{customdata[0]:.0f}<br>"
        "Total drug offense incidents: %{customdata[1]:.0f}<br>"
        "Marijuana rate: %{y:.2f}%<extra></extra>"
    )
))

fig.add_hline(y=0, line_dash='dash', line_color='black')

fig.update_layout(
    title=dict(text='Drug offense incidents over time', x=0.5),
    xaxis=dict(title=dict(text='Year'), automargin=True),
    yaxis=dict(title=dict(text='Number of incidents'), automargin=True),
    template='plotly_white',
    height=680,
    margin=dict(t=95, b=80, l=80, r=40),
    updatemenus=[
        dict(
            type='buttons',
            direction='right',
            x=0.0, y=1.07,
            xanchor='left', yanchor='top',
            showactive=True,
            active=0,
            buttons=[
                dict(
                    label='Marijuana rate view',
                    method='update',
                    args=[
                        {'visible': [False, False, True]},
                        {
                            'title.text': 'Marijuana rate within drug offense over time',
                            'xaxis.title.text': 'Year',
                            'yaxis.title.text': 'Marijuana share of drug offense (%)'
                        }
                    ]
                ),
                dict(
                    label='Incidents view',
                    method='update',
                    args=[
                        {'visible': [True, False, False]},
                        {
                            'title.text': 'Drug offense incidents over time',
                            'xaxis.title.text': 'Year',
                            'yaxis.title.text': 'Number of incidents'
                        }
                    ]
                ),
                dict(
                    label='Evolution view',
                    method='update',
                    args=[
                        {'visible': [False, True, False]},
                        {
                            'title.text': 'Drug offense incidents compared to the year before',
                            'xaxis.title.text': 'Year',
                            'yaxis.title.text': 'Change in percentage'
                        }
                    ]
                ),

            ]
        )
    ]
)

fig.show()
fig.write_html("drug_offense_interactive.html")

In [74]:
import folium
from folium.plugins import HeatMapWithTime
import pandas as pd

# Marijuana descriptions
marijuana = [
    'POSSESSION OF MARIJUANA',
    'POSSESSION OF MARIJUANA FOR SALES',
    'SALE OF MARIJUANA',
    'FURNISHING MARIJUANA',
    'TRANSPORTATION OF MARIJUANA',
    'PLANTING/CULTIVATING MARIJUANA',
    'Marijuana Offense',
    'Marijuana, Furnishing',
    'Marijuana, Cultivating/Planting',
    'Marijuana, Sales',
    'Marijuana, Transporting'
]

# Drug offense only, drop rows with missing coordinates
df_drug = df_fc[df_fc['incident_category'] == 'drug offense'].copy()
df_drug = df_drug.dropna(subset=['latitude', 'longitude'])
df_drug['year'] = df_drug['incident_date'].dt.year

# Split into marijuana vs other drug offense
df_mj = df_drug[df_drug['incident_description'].isin(marijuana)]
df_other = df_drug[~df_drug['incident_description'].isin(marijuana)]

# Sorted yearly time steps
years = sorted(df_drug['year'].unique().tolist())
year_labels = [str(int(y)) for y in years]

# Build one list-of-lists per year (each entry = [[lat, lon], ...])
mj_data = [df_mj[df_mj['year'] == y][['latitude', 'longitude']].values.tolist() for y in years]
other_data = [df_other[df_other['year'] == y][['latitude', 'longitude']].values.tolist() for y in years]

# Base map centred on San Francisco
m = folium.Map(location=[37.7749, -122.4194], zoom_start=12)

# Layer 1: Marijuana incidents — green gradient
HeatMapWithTime(
    data=mj_data,
    index=year_labels,
    gradient={0.2: '#90ee90', 0.5: '#228b22', 1.0: '#006400'},  # light green → dark green
    min_opacity=0.3,
    max_opacity=0.85,
    radius=12,
    blur=10,
    auto_play=False,
    display_index=True,
    name='Marijuana incidents'
).add_to(m)

# Layer 2: Other drug offense incidents — red/orange gradient
HeatMapWithTime(
    data=other_data,
    index=year_labels,
    gradient={0.2: '#ffff00', 0.5: '#ff8c00', 1.0: '#cc0000'},  # yellow → orange → red
    min_opacity=0.3,
    max_opacity=0.85,
    radius=12,
    blur=10,
    auto_play=False,
    display_index=True,
    name='Other drug offenses'
).add_to(m)

# Save
output_path = '/Users/venello/Repositories/Python/PyCharm/PycharmProjects/social_data_analysis/s195377.github.io/heatmap_interactive.html'
m.save(output_path)
print(f'Saved to {output_path}')
print(f'Years covered: {year_labels[0]} – {year_labels[-1]}')
print(f'Marijuana incidents total: {sum(len(f) for f in mj_data)}')
print(f'Other drug offense total:  {sum(len(f) for f in other_data)}')

Saved to /Users/venello/Repositories/Python/PyCharm/PycharmProjects/social_data_analysis/s195377.github.io/heatmap_interactive.html
Years covered: 2003 – 2025
Marijuana incidents total: 12021
Other drug offense total:  63595
